# 5B. Modelado Avanzado (XGBoost, LightGBM y Redes Neuronales)

## 1. Introducción
En el cuaderno anterior (5A) establecimos nuestra línea base de rendimiento mediante la optimización rigurosa de modelos tradicionales como la Regresión Logística y el Random Forest. Comprobamos que el fútbol es un entorno con muchísimo ruido estadístico, donde los modelos con mayor penalización matemática dominaron los resultados al evitar el sobreajuste.

En este cuaderno dejaremos atrás los algoritmos base y pondremos a prueba la verdadera capacidad predictiva de nuestro dataset utilizando algoritmos más avanzados de aprendizaje automático, los algoritmos de **Boosting** (XGBoost y LightGBM) y el **Aprendizaje Profundo** mediante Redes Neuronales Densas (MLP - Multilayer Perceptron). 

Al igual que en el cuaderno anterior, no nos conformaremos con configuraciones por defecto. Utilizaremos la técnica de **búsqueda en rejilla** para encontrar el equilibrio perfecto entre sesgo y varianza, explorando múltiples arquitecturas, profundidades y tasas de aprendizaje. Todo esto, por supuesto, manteniendo la validación fast-forward para garantizar que no se produzcan fugas de datos del futuro.

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import joblib  

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

os.makedirs('../models', exist_ok=True)

sys.path.append(os.path.abspath('../src'))

from model_utils import walk_forward_split, calculate_metrics

import warnings
warnings.filterwarnings('ignore')

import random

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

set_seed(42)


## 2. Carga de Datos
Al igual que en los cuadernos anteriores, excluimos de nuestro conjunto de datos aquellas columnas que no tienen valor predictivo o que supondrían una fuga de información (identificadores, fechas, nombres de equipos, goles anotados en el propio partido, etc.). Nos quedaremos únicamente con las métricas de rendimiento y estado de forma que calculamos en la fase de ingeniería de variables.

In [30]:
df = pd.read_csv('../data/processed/train_features.csv')

target_col = 'resultado_cod'
col_temporada = 'anio_fin_temporada'

cols_no_predictivas = [
    'id', 'fecha', 'local', 'visitante', 'division', col_temporada,
    'resultado', 'resultado_cod', 'goles_local', 'goles_visitante', 'puntos_local', 'puntos_visitante'
]

# XGBoost requiere que las clases empiecen en 0 y sean secuenciales
le = LabelEncoder()
df[target_col] = le.fit_transform(df[target_col])

## 3. Adaptadores para GridSearchCV
Dado que compartimos el mismo marco de evaluación que en el cuaderno de modelos básicos, nos enfrentamos exactamente a los mismos desafíos técnicos con la herramienta GridSearchCV. Por ello, volvemos a utilizar nuestros dos adaptadores personalizados:

In [31]:
def walk_forward_split_cruzado(df, col_temporada, window_type, window_size):
    """
    Generador que retorna Train (Todo) y Test (SOLO LaLiga).
    """
    base_splitter = walk_forward_split(df, col_temporada=col_temporada, window_type=window_type, window_size=window_size)
    
    for train_idx, test_idx in base_splitter:
        test_idx_laliga = df.loc[test_idx][df.loc[test_idx]['division'] == 'ESP_LaLiga'].index
        yield train_idx, test_idx_laliga

def auc_pr_scorer_wrapper(y_true, y_proba, **kwargs):
    """
    Envuelve nuestra función nativa de utilidades. Como GridSearchCV con needs_proba=True 
    solo pasa y_proba, inyectamos y_true como y_pred temporal (solo nos interesa el AUC-PR que devuelve).
    """
    _, auc_pr = calculate_metrics(y_true, y_pred=y_true, y_proba=y_proba)
    return auc_pr

auc_pr_scorer = make_scorer(auc_pr_scorer_wrapper, response_method='predict_proba')

## 4. Optimización para la Premier League

Procedemos a optimizar los hiperparámetros de nuestros algoritmos avanzados para la Premier League. 

Definimos un flujo de procesamiento de Scikit-Learn para:
* **Escalado de datos (StandardScaler):** A diferencia de los árboles de decisión, las **Redes Neuronales** exigen obligatoriamente que todos los datos de entrada operen en la misma escala (media 0 y varianza 1). Sin este paso previo, los gradientes de la red colapsarían y el modelo no lograría converger.
* **Selector de variables:** Mantenemos la opción de "passthrough" para evaluar el rendimiento puro de las variables, comparándolo nuevamente con los resultados que daría el uso de PCA.

### XGBoost y LightGBM
Ambos son los algoritmos dominantes en la actualidad para datos tabulares. Funcionan creando árboles de decisión de forma secuencial, donde cada nuevo árbol intenta corregir los errores residuales del anterior. LightGBM es la variante optimizada por Microsoft, diseñada para ser computacionalmente más ágil. En nuestra rejilla exploramos:
* **n_estimators:** El número total de ciclos de corrección (árboles).
* **learning_rate:** La tasa de aprendizaje. Un valor bajo hace que el modelo aprenda más despacio y requiera más árboles, pero genera predicciones mucho más robustas frente a datos no vistos.
* **max_depth:** Limita la profundidad de cada árbol para evitar que el algoritmo memorice el ruido inherente a los resultados del fútbol.

### Red Neuronal Densa (MLPClassifier)
Comprobaremos si una arquitectura de aprendizaje profundo es capaz de extraer relaciones no lineales en las rachas de los equipos que los modelos basados en árboles pasan por alto. Optimizamos:
* **hidden_layer_sizes:** La topología de la red (cantidad de capas ocultas y neuronas).
* **alpha:** Es el equivalente al parámetro C de la Regresión Logística (pero a la inversa), es decir, un "alpha" alto castiga fuertemente la complejidad de la red, forzándola a buscar patrones amplios y previniendo el sobreajuste.

### Exportación del mejor modelo
Guardamos, de nuevo, los modelos ganadores mediante la biblioteca joblib, generando archivos .pkl`en nuestro disco duro local para su posterior análisis e integración.

In [ ]:

df_premier = df[df['division'] == 'ENG_Premier'].copy().reset_index(drop=True)
X_prem = df_premier.drop(columns=cols_no_predictivas)
y_prem = df_premier[target_col]

cv_prem = list(walk_forward_split(df_premier, col_temporada=col_temporada, window_type='expanding', window_size=1))

# XGBoost
pipeline_xgb_prem = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', 'passthrough'),
    ('model', XGBClassifier(random_state=42, eval_metric='mlogloss'))
])

param_grid_xgb = {
    'selector': ['passthrough', PCA(n_components=0.95, random_state=42)],
    'model__n_estimators': [50, 100],
    'model__max_depth': [1, 3, 5],
    'model__learning_rate': [0.01, 0.1]
}

grid_xgb_prem = GridSearchCV(pipeline_xgb_prem, param_grid_xgb, cv=cv_prem, scoring=auc_pr_scorer, n_jobs=-1)
grid_xgb_prem.fit(X_prem, y_prem)

print(f"[XGBoost] Mejor AUC-PR: {grid_xgb_prem.best_score_:.4f}")
print(f"[XGBoost] Mejores Parámetros: {grid_xgb_prem.best_params_}\n")

# Red Neuronal Densa 
pipeline_mlp_prem = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', 'passthrough'),
    ('model', MLPClassifier(random_state=42, max_iter=500))
])

param_grid_mlp = {
    'selector': ['passthrough', PCA(n_components=0.95, random_state=42)],
    'model__hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'model__alpha': [0.0001, 0.001, 0.01]
}

grid_mlp_prem = GridSearchCV(pipeline_mlp_prem, param_grid_mlp, cv=cv_prem, scoring=auc_pr_scorer, n_jobs=-1)
grid_mlp_prem.fit(X_prem, y_prem)

print(f"[MLP] Mejor AUC-PR: {grid_mlp_prem.best_score_:.4f}")
print(f"[MLP] Mejores Parámetros: {grid_mlp_prem.best_params_}\n")


# LightGBM
pipeline_lgbm_prem = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('selector', 'passthrough'),
    ('model', LGBMClassifier(random_state=42, verbose=-1))
])

param_grid_lgbm = {
    'selector': ['passthrough', PCA(n_components=0.95, random_state=42)],
    'model__n_estimators': [50, 100],
    'model__max_depth': [3, 5],
    'model__learning_rate': [0.01, 0.1]
}

grid_lgbm_prem = GridSearchCV(pipeline_lgbm_prem, param_grid_lgbm, cv=cv_prem, scoring=auc_pr_scorer, n_jobs=-1)
grid_lgbm_prem.fit(X_prem, y_prem)

print(f"[LightGBM] Mejor AUC-PR: {grid_lgbm_prem.best_score_:.4f}")
print(f"[LightGBM ] Mejores Parámetros: {grid_lgbm_prem.best_params_}\n")

# Exportamos
if (grid_xgb_prem.best_score_ >= grid_mlp_prem.best_score_) and (grid_xgb_prem.best_score_ >= grid_lgbm_prem.best_score_):
    mejor_modelo_premier = grid_xgb_prem.best_estimator_
    nombre_archivo = '../models/xgboost_premier.pkl'
elif (grid_mlp_prem.best_score_ >= grid_xgb_prem.best_score_) and (grid_mlp_prem.best_score_ >= grid_lgbm_prem.best_score_):
    mejor_modelo_premier = grid_mlp_prem.best_estimator_
    nombre_archivo = '../models/mlp_premier.pkl'
else:
    mejor_modelo_premier = grid_lgbm_prem.best_estimator_
    nombre_archivo = '../models/lightgbm_premier.pkl'

joblib.dump(mejor_modelo_premier, nombre_archivo)

 OPTIMIZACIÓN PREMIER LEAGUE 
[XGBoost Premier] Mejor AUC-PR Cross-Val: 0.4627
[XGBoost Premier] Mejores Parámetros: {'model__learning_rate': 0.1, 'model__max_depth': 1, 'model__n_estimators': 100, 'selector': 'passthrough'}

[MLP Premier] Mejor AUC-PR Cross-Val: 0.4402
[MLP Premier] Mejores Parámetros: {'model__alpha': 0.0001, 'model__hidden_layer_sizes': (50,), 'selector': PCA(n_components=0.95, random_state=42)}

[LightGBM Premier] Mejor AUC-PR Cross-Val: 0.4534
[LightGBM Premier] Mejores Parámetros: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__n_estimators': 100, 'selector': 'passthrough'}



## 5. Optimización para LaLiga

Replicamos el proceso de optimización profunda, pero aplicando nuestra arquitectura asimétrica para LaLiga. 

Utilizamos el mismo flujo de procesamiento inicial de escalado y los mismos algoritmos (XGBoost, LightGBM y Red Neuronal), pero haciendo uso explícito de nuestro adaptador `cv_laliga` en el parámetro `cv` del GridSearchCV. De este modo, los modelos avanzados entrenarán absorbiendo todo el conocimiento táctico y estadístico de España e Inglaterra en conjunto, pero sus arquitecturas e hiperparámetros serán seleccionados estrictamente por su capacidad de acierto matemático sobre la liga española.

In [ ]:
print(" OPTIMIZACIÓN LALIGA ")

df_ambos = df.copy().reset_index(drop=True)
X_ambos = df_ambos.drop(columns=cols_no_predictivas)
y_ambos = df_ambos[target_col]

cv_laliga = list(walk_forward_split_cruzado(df_ambos, col_temporada=col_temporada, window_type='expanding', window_size=1))

# XGBoost
param_grid_xgb_laliga = {
    'selector': ['passthrough'],
    'model__n_estimators': [50, 100],
    'model__max_depth': [1, 3, 5],
    'model__learning_rate': [0.01, 0.1]
}

grid_xgb_laliga = GridSearchCV(pipeline_xgb_prem, param_grid_xgb_laliga, cv=cv_laliga, scoring=auc_pr_scorer, n_jobs=-1)
grid_xgb_laliga.fit(X_ambos, y_ambos)

print(f"[XGBoost] Mejor AUC-PR: {grid_xgb_laliga.best_score_:.4f}")
print(f"[XGBoost] Mejores Parámetros: {grid_xgb_laliga.best_params_}\n")

# Búsqueda para Red Neuronal Densa
param_grid_mlp_laliga = {
    'selector': ['passthrough'],
    'model__hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'model__alpha': [0.0001, 0.001, 0.01]
}

grid_mlp_laliga = GridSearchCV(pipeline_mlp_prem, param_grid_mlp_laliga, cv=cv_laliga, scoring=auc_pr_scorer, n_jobs=-1)
grid_mlp_laliga.fit(X_ambos, y_ambos)

print(f"[MLP] Mejor AUC-PR: {grid_mlp_laliga.best_score_:.4f}")
print(f"[MLP] Mejores Parámetros: {grid_mlp_laliga.best_params_}\n")


# LightGBM
param_grid_lgbm_laliga = {
    'selector': ['passthrough'],
    'model__n_estimators': [50, 100],
    'model__max_depth': [3, 5],
    'model__learning_rate': [0.01, 0.1]
}

grid_lgbm_laliga = GridSearchCV(pipeline_lgbm_prem, param_grid_lgbm_laliga, cv=cv_laliga, scoring=auc_pr_scorer, n_jobs=-1)
grid_lgbm_laliga.fit(X_ambos, y_ambos)

print(f"[LightGBM] Mejor AUC-PR: {grid_lgbm_laliga.best_score_:.4f}")
print(f"[LightGBM] Mejores Parámetros: {grid_lgbm_laliga.best_params_}\n")

# Exportamos
if (grid_xgb_laliga.best_score_ >= grid_mlp_laliga.best_score_) and (grid_xgb_laliga.best_score_ >= grid_lgbm_laliga.best_score_):
    mejor_modelo_laliga = grid_xgb_laliga.best_estimator_
    nombre_archivo = '../models/xgboost_laliga.pkl'
elif (grid_mlp_laliga.best_score_ >= grid_xgb_laliga.best_score_) and (grid_mlp_laliga.best_score_ >= grid_lgbm_laliga.best_score_):
    mejor_modelo_laliga = grid_mlp_laliga.best_estimator_
    nombre_archivo = '../models/mlp_laliga.pkl'
else:
    mejor_modelo_laliga = grid_lgbm_laliga.best_estimator_
    nombre_archivo = '../models/lightgbm_laliga.pkl'

joblib.dump(mejor_modelo_laliga, nombre_archivo)

 OPTIMIZACIÓN LALIGA 
[XGBoost LaLiga] Mejor AUC-PR Cross-Val (Medido en ESP): 0.4534
[XGBoost LaLiga] Mejores Parámetros: {'model__learning_rate': 0.1, 'model__max_depth': 1, 'model__n_estimators': 100, 'selector': 'passthrough'}

[MLP LaLiga] Mejor AUC-PR Cross-Val (Medido en ESP): 0.4300
[MLP LaLiga] Mejores Parámetros: {'model__alpha': 0.01, 'model__hidden_layer_sizes': (50,), 'selector': 'passthrough'}

[LightGBM LaLiga] Mejor AUC-PR Cross-Val (Medido en ESP): 0.4507
[LightGBM LaLiga] Mejores Parámetros: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__n_estimators': 100, 'selector': 'passthrough'}



## 6. Conclusión

La fase de modelado avanzado ha finalizado con éxito, generando los archivos .pkl definitivos. Sin embargo, al analizar los resultados y compararlos con los del Cuaderno 5A, nos encontramos ante uno de los fenómenos más interesantes del Machine Learning aplicado al deporte: **la paradoja de la complejidad**.

Contra todo pronóstico inicial, los algoritmos de vanguardia han obtenido un rendimiento inferior a nuestra Regresión Logística básica:
* En la **Premier League**, el modelo ganador de esta fase ha sido **XGBoost**, alcanzando un AUC-PR de **0.4627** (frente al 0.4853 de la Regresión Logística). El MLP y LightGBM se quedaron en 0.4402 y 0.4534 respectivamente.
* En el modelo global evaluado sobre **LaLiga**, el algoritmo más destacado ha vuelto a ser **XGBoost**, logrando una puntuación de **0.4534** (frente al 0.4723 de la Regresión Logística). El MLP y LightGBM marcaron 0.4300 y 0.4507.

El fútbol es un entorno con un altísimo componente estocástico, que significa que hay baja relación entre señal y ruido. Un partido puede decidirse por factores aleatorios (un rebote, una expulsión temprana). Los modelos como las Redes Neuronales o el Gradient Boosting tienen tanta capacidad para modelar relaciones no lineales que tienden a intentar buscarle una justificación matemática a esa aleatoriedad, cayendo irremediablemente en el sobreajuste. Por el contrario, la Regresión Logística actúa como un filtro más rígido, ya que capta la tendencia e ignora el ruido, generalizando mejor sobre datos futuros.

**Análisis de los hiperparámetros seleccionados:**
* **XGBoost:** El dato más revelador es que XGBoost ha seleccionado un `max_depth` de **1**. Un árbol de profundidad 1 es la arquitectura más simple posible. El propio optimizador ha detectado el exceso de ruido y ha capado la complejidad del árbol para intentar imitar el comportamiento de un modelo lineal simple.
* **MLP:** Ha sido el modelo con peor rendimiento en ambas ligas. Las redes neuronales requieren grandes volúmenes de datos con una señal clara para converger. En la Premier League, la red solo logró un mínimo rendimiento cuando el optimizador forzó la entrada del **PCA (0.95)**, demostrando que necesitaba reducir la dimensionalidad para intentar limpiar el ruido del dataset.
* **Mantenimiento del passthrough:** A excepción de la red neuronal en Inglaterra, los algoritmos de árboles han preferido utilizar todas las variables generadas en nuestra fase de ingeniería sin transformaciones, lo que vuelve a validar la calidad de las mismas.
